# Baseline Dataset Preparation and EDA

**Member 2 — Traditional ML Baselines**

This notebook prepares the FMA-small dataset (8,000 tracks, 8 balanced genres) for the classical ML baseline experiments described in the project reference. The baseline uses the MFCC feature subset from the official FMA feature extractor: 20 MFCC coefficients × 7 summary statistics = 140 features.

The workflow below uses the FMA-small metadata, the official precomputed feature table, a stratified 80/20 split with `random_state=42`, and `StandardScaler` fit only on the training set before evaluating four classical models.


## 1. Load FMA Metadata

In [ ]:
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data/fma_metadata")

tracks = pd.read_csv(
    DATA_DIR / "tracks.csv",
    index_col=0,
    header=[0, 1]
)

genres = pd.read_csv(
    DATA_DIR / "genres.csv"
)

print("Tracks shape:", tracks.shape)
print("Genres shape:", genres.shape)

print("\nTrack subsets:")
print(tracks[("set", "subset")].value_counts())

print("\nGenre columns:")
print(genres.columns.tolist())

Tracks shape: (106574, 52)
Genres shape: (163, 5)

Track subsets:
(set, subset)
large     81574
medium    17000
small      8000
Name: count, dtype: int64

Genre columns:
['genre_id', '#tracks', 'parent', 'title', 'top_level']


In [2]:
# Filter to FMA-small subset
small_tracks = tracks[tracks[("set", "subset")] == "small"]

print("Number of FMA-small tracks:", len(small_tracks))

print("\nGenre distribution:")
print(small_tracks[("track", "genre_top")].value_counts(dropna=False))

EXPECTED_GENRES = sorted(["Electronic", "Experimental", "Folk", "Hip-Hop",
                          "Instrumental", "International", "Pop", "Rock"])
actual_genres = sorted(small_tracks[("track", "genre_top")].dropna().unique().tolist())
print("\nExpected genres match:", actual_genres == EXPECTED_GENRES)
print("Genres:", actual_genres)

Number of FMA-small tracks: 8000

Genre distribution:
(track, genre_top)
Hip-Hop          1000
Pop              1000
Folk             1000
Experimental     1000
Rock             1000
International    1000
Electronic       1000
Instrumental     1000
Name: count, dtype: int64

Expected genres match: True
Genres: ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']


## 2. Load and Inspect Precomputed Features

In [3]:
# features.csv has a 3-level MultiIndex header: (feature, statistics, number)
# and track_id as the row index.  ~951 MB — loaded once.
features = pd.read_csv(
    DATA_DIR / "features.csv",
    index_col=0,
    header=[0, 1, 2]
)

print("Full features shape:", features.shape)
print(f"  Rows (tracks): {features.shape[0]:,}")
print(f"  Feature columns: {features.shape[1]}")
print(f"  Column header levels: {features.columns.nlevels}")
print(f"  Level names: {features.columns.names}")

print("\nFeature families:")
print(features.columns.get_level_values("feature").unique().tolist())

print("\nSummary statistics computed per feature:")
print(features.columns.get_level_values("statistics").unique().tolist())

print("\nFirst 10 column tuples:")
for col in features.columns[:10]:
    print(" ", col)

print("\nFirst 5 track IDs:", features.index[:5].tolist())

Full features shape: (106574, 518)
  Rows (tracks): 106,574
  Feature columns: 518
  Column header levels: 3
  Level names: ['feature', 'statistics', 'number']

Feature families:
['chroma_cens', 'chroma_cqt', 'chroma_stft', 'mfcc', 'rmse', 'spectral_bandwidth', 'spectral_centroid', 'spectral_contrast', 'spectral_rolloff', 'tonnetz', 'zcr']

Summary statistics computed per feature:
['kurtosis', 'max', 'mean', 'median', 'min', 'skew', 'std']

First 10 column tuples:
  ('chroma_cens', 'kurtosis', '01')
  ('chroma_cens', 'kurtosis', '02')
  ('chroma_cens', 'kurtosis', '03')
  ('chroma_cens', 'kurtosis', '04')
  ('chroma_cens', 'kurtosis', '05')
  ('chroma_cens', 'kurtosis', '06')
  ('chroma_cens', 'kurtosis', '07')
  ('chroma_cens', 'kurtosis', '08')
  ('chroma_cens', 'kurtosis', '09')
  ('chroma_cens', 'kurtosis', '10')

First 5 track IDs: [2, 3, 5, 10, 20]


## 3. Filter to FMA-Small

In [4]:
small_track_ids = small_tracks.index

# Baseline reference used in the assigned material is the MFCC-only feature subset.
# Official FMA feature extraction produces 518 total columns across families, but
# the classical baseline uses 20 MFCC coefficients × 7 moments = 140 features.
mfcc_columns = [col for col in features.columns if col[0] == "mfcc"]
print(f"Full FMA feature table: {features.shape}")
print(f"MFCC subset selected for baseline: {len(mfcc_columns)} columns")

# Verify coverage
present_mask = small_track_ids.isin(features.index)
n_present = present_mask.sum()
n_missing = (~present_mask).sum()
print(f"FMA-small tracks found in features: {n_present} / {len(small_track_ids)}")
print(f"Missing track IDs: {n_missing}")

# Filter features to FMA-small and keep only the baseline MFCC feature family
X_all = features.loc[small_track_ids, mfcc_columns]
# flatten MultiIndex columns to simple strings for ML compatibility
X_all.columns = ["_".join(col) for col in X_all.columns]

del features  # release the larger 518-column table

# Extract labels aligned by index
y_all = tracks.loc[small_track_ids, ("track", "genre_top")]

# Verify alignment
assert X_all.shape[0] == len(y_all), "Feature/label count mismatch"
assert (X_all.index == y_all.index).all(), "Feature/label index mismatch"
assert X_all.index.duplicated().sum() == 0, "Duplicate track IDs found"

print(f"\nFiltered feature matrix: {X_all.shape}")
print(f"Label vector: {y_all.shape}")
print(f"Unique labels: {sorted(y_all.unique().tolist())}")
print(f"Duplicate track IDs: {X_all.index.duplicated().sum()}")

print("\nClass distribution:")
print(y_all.value_counts().sort_index())


Full FMA feature table: (106574, 518)
MFCC subset selected for baseline: 140 columns
FMA-small tracks found in features: 8000 / 8000
Missing track IDs: 0

Filtered feature matrix: (8000, 140)
Label vector: (8000,)
Unique labels: ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']
Duplicate track IDs: 0

Class distribution:
(track, genre_top)
Electronic       1000
Experimental     1000
Folk             1000
Hip-Hop          1000
Instrumental     1000
International    1000
Pop              1000
Rock             1000
Name: count, dtype: int64


## 4. Feature Quality Checks

In [5]:
nan_total = X_all.isna().sum().sum()
inf_total = np.isinf(X_all.values).sum()

nan_per_col = X_all.isna().sum()
cols_with_nan = nan_per_col[nan_per_col > 0]

nunique = X_all.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()

print("=== Feature Quality Report ===")
print(f"Total NaN values:      {nan_total}")
print(f"Total Inf values:      {inf_total}")
print(f"Columns with NaN:      {len(cols_with_nan)}")
print(f"Constant columns:      {len(constant_cols)}")

if len(cols_with_nan) > 0:
    print("\nColumns with NaN values:")
    print(cols_with_nan)

if len(constant_cols) > 0:
    print("\nConstant columns (will be dropped before training):")
    print(constant_cols)

if nan_total == 0 and inf_total == 0 and len(constant_cols) == 0:
    print("\n=> All features are clean. No imputation or removal needed.")

=== Feature Quality Report ===
Total NaN values:      0
Total Inf values:      0
Columns with NaN:      0
Constant columns:      0

=> All features are clean. No imputation or removal needed.


## 5. Exploratory Data Analysis

In [6]:
print("=== Dataset Summary ===")
print(f"Total samples:         {X_all.shape[0]:,}")
print(f"Feature dimensions:    {X_all.shape[1]}")
print(f"Number of classes:     {y_all.nunique()}")
print(f"Samples per class:     {X_all.shape[0] // y_all.nunique()}")
print(f"Missing values:        {nan_total}")
print(f"Infinite values:       {inf_total}")

# Class distribution bar chart
class_counts = y_all.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
colors = plt.cm.Set2(np.linspace(0, 1, len(class_counts)))
bars = ax.bar(class_counts.index, class_counts.values, color=colors, edgecolor="#333", linewidth=0.6)
ax.set_xlabel("Genre", fontsize=11)
ax.set_ylabel("Number of Tracks", fontsize=11)
ax.set_title("FMA-Small: Class Distribution (8 Genres × 1,000 Tracks)", fontsize=13, fontweight="bold")
ax.set_ylim(0, max(class_counts.values) * 1.15)

for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            str(count), ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=120)
plt.show()
print("Saved: class_distribution.png")

=== Dataset Summary ===
Total samples:         8,000
Feature dimensions:    140
Number of classes:     8
Samples per class:     1000
Missing values:        0
Infinite values:       0
Saved: class_distribution.png


C:\Users\Trisha V P\AppData\Local\Temp\ipykernel_11836\4234851141.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Feature-level summary: count of features per audio feature family
# Column names follow the pattern: {feature}_{statistic}_{number}
feature_families = [col.rsplit("_", 2)[0] for col in X_all.columns]
family_counts = pd.Series(feature_families).value_counts().sort_index()

print("Feature columns per audio feature family:")
print(family_counts)
print(f"\nTotal feature columns: {family_counts.sum()}")

Feature columns per audio feature family:
mfcc    140
Name: count, dtype: int64

Total feature columns: 140


## 6. Train/Test Split and Preprocessing

In [8]:
X = X_all.values.astype(np.float64)
y = y_all.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=== Train/Test Split ===")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

# Verify stratification
print("\nTrain class distribution:")
train_counts = pd.Series(y_train).value_counts().sort_index()
print(train_counts)

print("\nTest class distribution:")
test_counts = pd.Series(y_test).value_counts().sort_index()
print(test_counts)

=== Train/Test Split ===
X_train: (6400, 140)
X_test:  (1600, 140)
y_train: (6400,)
y_test:  (1600,)

Train class distribution:
Electronic       800
Experimental     800
Folk             800
Hip-Hop          800
Instrumental     800
International    800
Pop              800
Rock             800
Name: count, dtype: int64

Test class distribution:
Electronic       200
Experimental     200
Folk             200
Hip-Hop          200
Instrumental     200
International    200
Pop              200
Rock             200
Name: count, dtype: int64


In [9]:
# StandardScaler: fit on training data ONLY to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=== Scaling Verification ===")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")
print(f"\nTrain mean (should be ~0): {X_train_scaled.mean():.6f}")
print(f"Train std  (should be ~1): {X_train_scaled.std():.6f}")
print(f"Test mean  (close to 0):   {X_test_scaled.mean():.6f}")
print(f"Test std   (close to 1):   {X_test_scaled.std():.6f}")

print("\n=== Data Leakage Check ===")
print("Scaler fitted on X_train only:", scaler.n_samples_seen_ == X_train.shape[0])
print(f"Scaler saw {scaler.n_samples_seen_} samples (train size: {X_train.shape[0]})")

=== Scaling Verification ===
X_train_scaled shape: (6400, 140)
X_test_scaled shape:  (1600, 140)

Train mean (should be ~0): 0.000000
Train std  (should be ~1): 1.000000
Test mean  (close to 0):   0.005553
Test std   (close to 1):   0.989011

=== Data Leakage Check ===
Scaler fitted on X_train only: True
Scaler saw 6400.0 samples (train size: 6400)


In [10]:
print("="*50)
print("MILESTONE COMPLETE: Dataset Preparation")
print("="*50)
print(f"\nTotal samples:       {X.shape[0]:,}")
print(f"Number of features:  {X.shape[1]}")
print(f"Number of classes:   {len(np.unique(y))}")
print(f"Samples per class:   {X.shape[0] // len(np.unique(y))}")
print(f"\nTrain set:           {X_train.shape[0]:,} samples")
print(f"Test set:            {X_test.shape[0]:,} samples")
print(f"\nNaN values:          {nan_total}")
print(f"Inf values:          {inf_total}")
print(f"Constant columns:    {len(constant_cols)}")
print(f"\nScaling:             StandardScaler (fit on train only)")
print(f"Split:               80/20, stratified, random_state=42")
print(f"\nAvailable arrays:")
print(f"  Unscaled: X_train {X_train.shape}, X_test {X_test.shape}")
print(f"  Scaled:   X_train_scaled {X_train_scaled.shape}, X_test_scaled {X_test_scaled.shape}")
print(f"  Labels:   y_train {y_train.shape}, y_test {y_test.shape}")
print(f"\nNo ML models have been trained.")
print(f"Ready for model training in next milestone.")

MILESTONE COMPLETE: Dataset Preparation

Total samples:       8,000
Number of features:  140
Number of classes:   8
Samples per class:   1000

Train set:           6,400 samples
Test set:            1,600 samples

NaN values:          0
Inf values:          0
Constant columns:    0

Scaling:             StandardScaler (fit on train only)
Split:               80/20, stratified, random_state=42

Available arrays:
  Unscaled: X_train (6400, 140), X_test (1600, 140)
  Scaled:   X_train_scaled (6400, 140), X_test_scaled (1600, 140)
  Labels:   y_train (6400,), y_test (1600,)

No ML models have been trained.
Ready for model training in next milestone.


# 7. Classical ML Baselines

In this section, we train and evaluate four classical machine learning baselines for 8-genre music classification using the FMA-small dataset and the MFCC-only baseline feature set.

1. **K-Nearest Neighbors (KNN)**: An instance-based non-parametric classifier that assigns genres based on Euclidean distance in the 140-dimensional MFCC feature space.
2. **Logistic Regression**: A convex linear probabilistic model that establishes a transparent linear decision boundary.
3. **Linear Support Vector Machine (LinearSVC)**: A maximum-margin linear classifier designed for robust generalization in high-dimensional sparse or dense feature spaces.
4. **Multi-Layer Perceptron (MLP)**: A feedforward artificial neural network with non-linear activation, included to benchmark a small neural baseline against the linear models.

### Experimental Setup & Reproducibility:
- **Dataset**: FMA-small subset with 8 balanced genres and 8,000 tracks.
- **Feature representation**: 140 MFCC features, corresponding to the official FMA `mfcc` family: 20 MFCC coefficients × 7 statistics (`mean`, `std`, `skew`, `kurtosis`, `median`, `min`, `max`).
- **Leakage-free preprocessing**: `StandardScaler` is fit only on the training split; the test split is transformed afterward.
- **Strict reproducibility**: `random_state=42` is fixed across all stochastic baselines.
- **Evaluation metrics**: Accuracy, macro precision, macro recall, macro F1, and training/inference time.
- **Comparison outputs**: Results table, confusion matrices, and a concise interpretation.


In [11]:
import time
import warnings
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Suppress benign scikit-learn warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Fixed genre ordering for consistent evaluation across all models
GENRE_ORDER = [
    "Electronic",
    "Experimental",
    "Folk",
    "Hip-Hop",
    "Instrumental",
    "International",
    "Pop",
    "Rock"
]

# Initialize dictionaries to store models, predictions, and evaluation records
baseline_models = {
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Linear Support Vector Machine": LinearSVC(random_state=42, max_iter=5000, dual="auto"),
    "Multi-Layer Perceptron": MLPClassifier(
        hidden_layer_sizes=(128,),
        max_iter=100,
        early_stopping=True,
        random_state=42
    )
}

baseline_predictions = {}
results_records = []

In [12]:
# Train and evaluate each baseline model
for name, model in baseline_models.items():
    print("=" * 65)
    print(f"Model: {name}")
    print("=" * 65)
    
    # 1. Train model on scaled training features
    t_train_start = time.perf_counter()
    model.fit(X_train_scaled, y_train)
    t_train = time.perf_counter() - t_train_start
    
    # 2. Predict on scaled test features
    t_pred_start = time.perf_counter()
    y_pred = model.predict(X_test_scaled)
    t_pred = time.perf_counter() - t_pred_start
    
    # Store predictions for later reuse
    baseline_predictions[name] = y_pred
    
    # 3. Calculate evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    macro_prec = precision_score(y_test, y_pred, labels=GENRE_ORDER, average="macro", zero_division=0)
    macro_rec = recall_score(y_test, y_pred, labels=GENRE_ORDER, average="macro", zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, labels=GENRE_ORDER, average="macro", zero_division=0)
    
    results_records.append({
        "Model": name,
        "Accuracy": acc,
        "Macro Precision": macro_prec,
        "Macro Recall": macro_rec,
        "Macro F1": macro_f1,
        "Training Time (s)": round(t_train, 3),
        "Prediction Time (s)": round(t_pred, 3)
    })
    
    print(f"Training Time:   {t_train:.3f} s")
    print(f"Prediction Time: {t_pred:.3f} s")
    print(f"Test Accuracy:   {acc:.4f} ({acc*100:.2f}%)")
    print(f"Macro F1-Score:  {macro_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, labels=GENRE_ORDER, digits=4, zero_division=0))
    print()


Model: K-Nearest Neighbors
Training Time:   0.019 s
Prediction Time: 0.264 s
Test Accuracy:   0.5012 (50.12%)
Macro F1-Score:  0.4895

Classification Report:
               precision    recall  f1-score   support

   Electronic     0.4410    0.4300    0.4354       200
 Experimental     0.4324    0.2400    0.3087       200
         Folk     0.5360    0.7450    0.6234       200
      Hip-Hop     0.4636    0.6050    0.5249       200
 Instrumental     0.5459    0.5350    0.5404       200
International     0.5323    0.7000    0.6048       200
          Pop     0.3598    0.2950    0.3242       200
         Rock     0.6970    0.4600    0.5542       200

     accuracy                         0.5012      1600
    macro avg     0.5010    0.5012    0.4895      1600
 weighted avg     0.5010    0.5012    0.4895      1600


Model: Logistic Regression
Training Time:   51.071 s
Prediction Time: 0.002 s
Test Accuracy:   0.5100 (51.00%)
Macro F1-Score:  0.5007

Classification Report:
               prec

In [13]:
# Generate confusion matrices for all four models in a clean 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(15, 13))
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(baseline_predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=GENRE_ORDER)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=GENRE_ORDER)
    disp.plot(
        ax=axes[idx],
        cmap="Blues",
        values_format="d",
        colorbar=False
    )
    model_acc = [r["Accuracy"] for r in results_records if r["Model"] == name][0]
    axes[idx].set_title(f"{name}\nAccuracy: {model_acc:.2%}", fontsize=12, fontweight="bold")
    axes[idx].set_xticklabels(GENRE_ORDER, rotation=35, ha="right", fontsize=9)
    axes[idx].set_yticklabels(GENRE_ORDER, fontsize=9)

plt.suptitle("Classical ML Baselines: Confusion Matrices (FMA-Small Test Set, N=1,600)", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.savefig("baseline_confusion_matrices.png", dpi=120)
plt.show()
print("Saved: baseline_confusion_matrices.png")


Saved: baseline_confusion_matrices.png


C:\Users\Trisha V P\AppData\Local\Temp\ipykernel_11836\3077637656.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Construct comparison DataFrame
baseline_results = pd.DataFrame(results_records)

# Sort by Accuracy descending as presentation convenience
baseline_results = baseline_results.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print("=" * 75)
print("BASELINE MODELS COMPARISON TABLE")
print("=" * 75)
display_table = baseline_results.copy()
for col in ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]:
    display_table[col] = display_table[col].apply(lambda v: f"{v:.4f} ({v*100:.2f}%)")
for col in ["Training Time (s)", "Prediction Time (s)"]:
    display_table[col] = display_table[col].apply(lambda v: f"{v:.3f}")
print(display_table.to_string(index=False))

# Select the strongest model from measured test accuracy
best_model_name = baseline_results.loc[baseline_results["Accuracy"].idxmax(), "Model"]
best_model_row = baseline_results.loc[baseline_results["Accuracy"].idxmax()]
best_y_pred = baseline_predictions[best_model_name]

# Accuracy comparison
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]
bars = ax.bar(baseline_results["Model"], baseline_results["Accuracy"], color=colors, edgecolor="#333", linewidth=0.7)
ax.set_title("Classical ML Baselines: Test Accuracy Comparison (FMA-Small)", fontsize=13, fontweight="bold")
ax.set_ylabel("Test Accuracy", fontsize=11)
ax.set_ylim(0, max(baseline_results["Accuracy"]) * 1.18)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.01, f"{height:.2%}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.axhline(0.125, color="gray", linestyle="--", linewidth=1, label="Random Baseline (12.5%)")
ax.legend(loc="upper right")
plt.xticks(rotation=15, ha="right", fontsize=10)
plt.tight_layout()
plt.savefig("baseline_accuracy_comparison.png", dpi=120)
plt.show()
print("Saved: baseline_accuracy_comparison.png")

# Macro F1 comparison
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(baseline_results["Model"], baseline_results["Macro F1"], color=colors, edgecolor="#333", linewidth=0.7)
ax.set_title("Classical ML Baselines: Macro F1 Comparison (FMA-Small)", fontsize=13, fontweight="bold")
ax.set_ylabel("Macro F1 Score", fontsize=11)
ax.set_ylim(0, max(baseline_results["Macro F1"]) * 1.18)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.01, f"{height:.2%}", ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.xticks(rotation=15, ha="right", fontsize=10)
plt.tight_layout()
plt.savefig("baseline_macro_f1_comparison.png", dpi=120)
plt.show()
print("Saved: baseline_macro_f1_comparison.png")

# Confusion matrix for the best model by test accuracy
best_cm = confusion_matrix(y_test, best_y_pred, labels=GENRE_ORDER)
fig, ax = plt.subplots(figsize=(9, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=best_cm, display_labels=GENRE_ORDER)
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title(f"Best Baseline by Accuracy: {best_model_name}\nAccuracy: {best_model_row['Accuracy']:.2%}", fontsize=13, fontweight="bold")
ax.set_xticklabels(GENRE_ORDER, rotation=35, ha="right", fontsize=10)
ax.set_yticklabels(GENRE_ORDER, fontsize=10)
plt.tight_layout()
plt.savefig("baseline_best_model_confusion_matrix.png", dpi=120)
plt.show()
print("Saved: baseline_best_model_confusion_matrix.png")

# Final measured result and interpretation
print("\n=== BEST BASELINE SUMMARY ===")
print(f"Selected by test accuracy: {best_model_name}")
print(f"Accuracy: {best_model_row['Accuracy']:.4f} ({best_model_row['Accuracy'] * 100:.2f}%)")
print(f"Macro Precision: {best_model_row['Macro Precision']:.4f} ({best_model_row['Macro Precision'] * 100:.2f}%)")
print(f"Macro Recall: {best_model_row['Macro Recall']:.4f} ({best_model_row['Macro Recall'] * 100:.2f}%)")
print(f"Macro F1: {best_model_row['Macro F1']:.4f} ({best_model_row['Macro F1'] * 100:.2f}%)")
print(f"Evaluation set: {len(y_test):,} held-out tracks across {len(GENRE_ORDER)} genres")
print(f"Input representation: {X.shape[1]} MFCC summary features")
print("Interpretation: this identifies the strongest of the four baselines on this fixed split; it is not a cross-validation estimate.")

BASELINE MODELS COMPARISON TABLE
                        Model        Accuracy Macro Precision    Macro Recall        Macro F1 Training Time (s) Prediction Time (s)
       Multi-Layer Perceptron 0.5437 (54.37%) 0.5375 (53.75%) 0.5437 (54.37%) 0.5394 (53.94%)             8.204               0.002
          Logistic Regression 0.5100 (51.00%) 0.4970 (49.70%) 0.5100 (51.00%) 0.5007 (50.07%)            51.071               0.002
Linear Support Vector Machine 0.5081 (50.81%) 0.4865 (48.65%) 0.5081 (50.81%) 0.4825 (48.25%)            10.094               0.001
          K-Nearest Neighbors 0.5012 (50.12%) 0.5010 (50.10%) 0.5012 (50.12%) 0.4895 (48.95%)             0.019               0.264


C:\Users\Trisha V P\AppData\Local\Temp\ipykernel_11836\3444745515.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: baseline_accuracy_comparison.png


C:\Users\Trisha V P\AppData\Local\Temp\ipykernel_11836\3444745515.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: baseline_macro_f1_comparison.png
Saved: baseline_best_model_confusion_matrix.png

=== BEST BASELINE SUMMARY ===
Selected by test accuracy: Multi-Layer Perceptron
Accuracy: 0.5437 (54.37%)
Macro Precision: 0.5375 (53.75%)
Macro Recall: 0.5437 (54.37%)
Macro F1: 0.5394 (53.94%)
Evaluation set: 1,600 held-out tracks across 8 genres
Input representation: 140 MFCC summary features
Interpretation: this identifies the strongest of the four baselines on this fixed split; it is not a cross-validation estimate.


C:\Users\Trisha V P\AppData\Local\Temp\ipykernel_11836\3444745515.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
print("=" * 60)
print("MILESTONE COMPLETE: Classical ML Baselines")
print("=" * 60)
print(f"Models trained & evaluated: {len(baseline_results)}")
print(f"Stored in baseline_models:   {list(baseline_models.keys())}")
print(f"Stored in baseline_predictions: {list(baseline_predictions.keys())}")
print("\nFinal Results Ranked by Accuracy:")
for rank, row in baseline_results.iterrows():
    print(f"  {rank+1}. {row['Model']:<30} Accuracy: {row['Accuracy']:.4f} | Macro F1: {row['Macro F1']:.4f} | Train Time: {row['Training Time (s)']:.3f}s")

strongest = baseline_results.loc[baseline_results["Accuracy"].idxmax()]
print(f"\nStrongest baseline by Accuracy: {strongest['Model']} ({strongest['Accuracy']:.4f}, Macro F1={strongest['Macro F1']:.4f})")
print("\nAll baselines successfully executed. Comparison table ready.")


MILESTONE COMPLETE: Classical ML Baselines
Models trained & evaluated: 4
Stored in baseline_models:   ['K-Nearest Neighbors', 'Logistic Regression', 'Linear Support Vector Machine', 'Multi-Layer Perceptron']
Stored in baseline_predictions: ['K-Nearest Neighbors', 'Logistic Regression', 'Linear Support Vector Machine', 'Multi-Layer Perceptron']

Final Results Ranked by Accuracy:
  1. Multi-Layer Perceptron         Accuracy: 0.5437 | Macro F1: 0.5394 | Train Time: 8.204s
  2. Logistic Regression            Accuracy: 0.5100 | Macro F1: 0.5007 | Train Time: 51.071s
  3. Linear Support Vector Machine  Accuracy: 0.5081 | Macro F1: 0.4825 | Train Time: 10.094s
  4. K-Nearest Neighbors            Accuracy: 0.5012 | Macro F1: 0.4895 | Train Time: 0.019s

Strongest baseline by Accuracy: Multi-Layer Perceptron (0.5437, Macro F1=0.5394)

All baselines successfully executed. Comparison table ready.
